# Årliga Medelvärden och Korrelationsanalys
Denna notebook beräknar och visualiserar det årliga medelvärdet per index (NDVI, NDWI, MNDWI, NDMI, EVI) för givna studieområden. Vidare genomförs en statistisk bedömning av hur starkt dessa index korrelerar med varandra över tid med hjälp av Spearmans korrelationsanalys.

### Importer
Importerar nödvändiga databehandlings- och visualiseringsbibliotek (`numpy`, `pandas`, `geopandas`, `matplotlib`, `seaborn`) samt nödvändiga konfigureringar från `config.py`.

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from config import (
    build_analysis_output_dirs,
    build_analysis_satellites,
    build_stack,
    load_and_prepare_scene_logs,
    print_satellite_setup,
    OPEN_WETLAND_AREA,
    select_best_scenes,
    select_all_clear_scenes,
    )

### Konfiguration och Initiering
Här ställs in vilket studieområde och satellitsensor som ska användas. Koden hanterar också automatiskt en mappning av namn (till exempel att "TAM" i grafer/filer refereras till som "REF"). Filkataloger för att spara utdata skapas upp vid behov.

In [ ]:
# ============================================
# Konfigurera sökvägar och studieområde
# ============================================

area = "TAM"  # OBS: här väljs studieområdet
sensor = "landsat"  # "landsat" eller "sentinel2"

if area == "TAM":
    name = "REF"
elif area == "GM":
    name = "RH"
elif area == "LF":
    name = "HB"
else:
    name = area

satellites = build_analysis_satellites(area, sensor=sensor)

analysis_output_dirs = build_analysis_output_dirs(sensor=sensor)
output_dir_change_images = analysis_output_dirs["change_images"]
os.makedirs(output_dir_change_images, exist_ok=True)

print_satellite_setup(area, satellites, sensor=sensor)

### Steg 1: Inläsning av scenloggar
Metadata-loggen (`log`) från preprocessingen hämtas in för att scriptet ska veta vilka förprocessade rasterbilder som finns att tillgå.

In [ ]:
# =============================================
# Steg 1: Läs in och kombinera loggar
# =============================================
log = load_and_prepare_scene_logs(satellites, area)
log = log.sort_values("date").reset_index(drop=True)

In [ ]:
# Valbart inför körning: sätt True för att exkludera enbart Landsat 7
exclude_l07_this_run = True

if exclude_l07_this_run:
    before_n = len(log)
    if "sat_name" in log.columns:
        # Strikt filter: ta bara bort L07-rader, behåll L05/L0809
        log = log[log["sat_name"].astype(str).str.upper() != "L07"].copy()
    elif "scene_name" in log.columns:
        # Fallback om sat_name saknas i loggen
        log = log[~log["scene_name"].astype(str).str.contains("LE07", case=False, na=False)].copy()
    print(f"L07 exkluderat denna körning: {before_n - len(log)} scener borttagna")


### Steg 2: Urval av aktuella scener
Här görs ett urval – exempelvis den molnfriaste bilden närmast den 15:e Juli för varje ingående år. Detta bygger en konsekvent referens över tid utan bruskälla från alltför stor säsongsvariation.

In [ ]:
# =============================================
# Steg 2: Välj molnfria scener (alla eller en per år)
# =============================================

# Använd för Sentinel-2
# clear_scenes = select_all_clear_scenes(log) # Väljer alla molnfria scener
#print(f"Antal valda molnfria scener: {len(clear_scenes)}")
#print(clear_scenes[["year", "date", "days_from_july15", "sat_name", "cloud_pct"]])

# Använd för Landsat
best_scenes = select_best_scenes(log) # Väljer en bästa molnfri scen per år närmast 15 juli
print(f"Antal valda bästa scener: {len(best_scenes)}")
print(best_scenes[["year", "date", "days_from_july15", "sat_name", "cloud_pct"]])

### Steg 3: Bygga Datakuber (stacks)
Koden laddar pixel för pixel för samtliga tilldelade satellitbilder och bygger upp en rumslig xarray-stack per valt index (NDVI, NDWI, osv). Denna stack underlättar tunga medelvärdesberäkningar avsevärt.

In [ ]:
# =============================================
# Steg 3: Bygg xarray-stack per index CLEAR scenes
# =============================================

# Använd för Sentinel-2, skapar stacks från alla molnfria scener. 
# stacks = {}
# for index_name in ["MNDWI","EVI"]:
#     print(f"Bygger stack för {index_name}...")
#     stacks[index_name] = build_stack(clear_scenes, index_name, area)
#     print(f"  Shape: {stacks[index_name].shape}")


# Använd för Landsat och medelvärden, skapar stacks från bästa scen per år.
stacks = {}
for index_name in ["MNDWI","EVI","NDVI","NDWI","NDMI"]:
    print(f"Bygger stack för {index_name}...")
    stacks[index_name] = build_stack(best_scenes, index_name, area)
    print(f"  Shape: {stacks[index_name].shape}")

## Visualisering - medelvärden

### Steg 4: Årligt medelvärde och tidsseriegraf
Variablerna i stacken genomsnittberäknas över intresseområdet (definerat per shapefil). Både diagrammet (PNG) och den bakomliggande datan (CSV) exporteras i slutändan till mappen för "Change Images".

In [ ]:
# =============================================
# Steg 5: Årligt medelvärde per index inom studieområdet
# =============================================
from scipy.interpolate import pchip_interpolate

# Dynamiskt val av locations: east/west för GRM och LF, annars "all"
is_split_area = area in ["GRM", "LF"]
if is_split_area:
    locations = ["east", "west"]
else:
    locations = ["all"]

# Unika färger per index, olika nyanser för east och west men ljusare för "all"
colors = {
    "MNDWI": {"east": "darkblue", "west": "cornflowerblue", "all": "steelblue"},
    "EVI": {"east": "darkgreen", "west": "yellowgreen", "all": "forestgreen"},
    "NDVI": {"east": "darkred", "west": "lightcoral", "all": "firebrick"},
    "NDMI": {"east": "darkgoldenrod", "west": "palegoldenrod", "all": "goldenrod"},
    "NDWI": {"east": "darkmagenta", "west": "plum", "all": "orchid"}
}
# Båda ska ha heldragen linje och runda punkter
loc_styles = {
    "east": {"linestyle": "-", "marker": "o"},
    "west": {"linestyle": "-", "marker": "o"},
    "all": {"linestyle": "-", "marker": "o"}
}

# Skapa diagrammet i lite mindre bredd och med högre upplösning (dpi=600)
fig, ax = plt.subplots(figsize=(7, 3), dpi=600)
rows = []

for loc in locations:
    if loc in ["east", "west"]:
        clip_key = f"{area}_wetland_{loc}"
    else:
        clip_key = f"{area}_wetland"
    
    if clip_key not in OPEN_WETLAND_AREA:
        print(f"Varning: Hittade ingen filepath för {clip_key} i OPEN_WETLAND_AREA. Hoppar över.")
        continue
        
    clip_shp = OPEN_WETLAND_AREA[clip_key]
    if clip_shp is not None and os.path.exists(clip_shp):
        aoi = gpd.read_file(clip_shp)
    else:
        print(f"Varning: Kunde inte ladda {clip_shp}. Kontrollera att filen finns.")
        continue

    for index_name, stack in stacks.items():
        aoi_proj = aoi.to_crs(stack.rio.crs)
        
        # Gruppera på år oavsett sensor (hanterar flera scener per år)
        yearly_stack = stack.groupby("time.year").mean(dim="time")
        years = yearly_stack["year"].values
        s = yearly_stack.rio.clip(aoi_proj.geometry, aoi_proj.crs)

        mean_vals = s.mean(dim=["x", "y"], skipna=True)
        
        # Inställningar för just denna kombination av index och lokation
        if loc in ["east", "west"]:
            label_name = f"{index_name} ({loc.capitalize()})"
        else:
            label_name = index_name
            
        base_color = colors.get(index_name, {}).get(loc, "gray")
        style = loc_styles[loc]
        
        # Skapa en utjämnad originallinje med pchip
        if len(years) > 3:
            years_smooth = np.linspace(years.min(), years.max(), 300)
            vals_smooth = pchip_interpolate(years, mean_vals.values, years_smooth)
            
            # Plotta smoothad linje
            ax.plot(years_smooth, vals_smooth, linewidth=1.5, linestyle=style["linestyle"], 
                    color=base_color, label=label_name)
            # Sätt fasta punkter över åren
            ax.scatter(years, mean_vals.values, marker=style["marker"], s=15, 
                       color=base_color, zorder=3)
        else:
            # Mindre än 4 år, fall back till vanlig plotting
            ax.plot(years, mean_vals.values, marker=style["marker"], linestyle=style["linestyle"], 
                    markersize=4, linewidth=1.5, color=base_color, label=label_name)
        
        # Samla data för CSV
        for year, val in zip(years, mean_vals.values):
            rows.append({
                "area":       area,
                "location":   loc if is_split_area else "main",
                "index":      index_name,
                "year":       year,
                "mean_value": val
            })

# ==========================================
# STYLING (diagrammets utseende)
# ==========================================

# Svaga streckade grå linjer vid varje tickmark på y-axeln
ax.grid(True, axis='y', linestyle='--', linewidth=0.5, color='gray', alpha=0.5, zorder=0)


ax.set_ylabel("Mean Index Value", fontsize=9) # Mindre font

# Formatera ticks: Inåtriktade, tunnare och mindre font
ax.tick_params(axis='both', which='major', direction='in', length=4, width=1, labelsize=9)

# Sätt ticks på x-axeln enligt de specifika årtalen
if len(rows) > 0:
    if sensor == "sentinel2":
        ax.set_xticks(range(2015, 2026))
        # Mindre marginal så att det inte blir för mycket tomrum innan 2015
        ax.set_xlim(2014.5, 2025.5)
    else:
        ax.set_xticks([1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025])
        # För att inte datapunkter år 1984 eller 2025 ska hamna precis på kanten
        ax.set_xlim(1983, 2026)
        
    ax.set_ylim(bottom=-0.65, top=0.65)  # Justera y-lim för att ge lite mer utrymme ovanför 1.0

# Ta bort övre och högra diagramlinjen (svarta bocken) 
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Behåll botten och vänster men gör de lite tunnare
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

# Hämta handles och labels för legend och sortera dem i önskad ordning
handles, labels = ax.get_legend_handles_labels()

# Önskade namn och deras ordning
if is_split_area:
    # 5 kolumner med West på övre raden och East på nedre raden
    order = [
        "NDVI (West)", "NDVI (East)", "EVI (West)", "EVI (East)", "NDMI (West)",
        "NDMI (East)", "MNDWI (West)", "MNDWI (East)", "NDWI (West)", "NDWI (East)",   
    ]
else:
    order = ["NDVI", "EVI",  "NDMI", "MNDWI", "NDWI"]

# Mappa matchande handles enligt ordningen
ordered_handles = []
ordered_labels = []
for label in order:
    if label in labels:
        idx = labels.index(label)
        ordered_handles.append(handles[idx])
        ordered_labels.append(labels[idx])
        
# Lägg till eventuella övriga (om de saknades i ordningslistan)
for h, l in zip(handles, labels):
    if l not in ordered_labels:
        ordered_handles.append(h)
        ordered_labels.append(l)

# Teckenförklaring placerad under formatet igen
cols = 5
if ordered_handles:
    ax.legend(ordered_handles, ordered_labels, loc='upper center', bbox_to_anchor=(0.5, -0.2), 
              ncol=cols, frameon=False, fontsize=8)
else:
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), 
              ncol=cols, frameon=False, fontsize=8)

plt.tight_layout()

# Spara diagrammet som en högupplöst PNG-bild (bäst för grafer)
suffix = "_east_west" if is_split_area else ""
fig_path = os.path.join(output_dir_change_images, f"{name}{suffix}_mean_values_{sensor}_REPORTareas.png")
fig.savefig(fig_path, dpi=600, bbox_inches='tight', facecolor='white')
print(f"Bild sparad: {fig_path}")

plt.show()

if rows:
    # Spara som CSV
    df = pd.DataFrame(rows)
    out_path = os.path.join(output_dir_change_images, f"{name}{suffix}_mean_values_{sensor}_REPORTareas.csv")
    df.to_csv(out_path, index=False)
    print(f"Sparad: {out_path}")
else:
    print("Ingen data samlades in, sparar ingen CSV.")

### Steg 5: Korrelationsanalys mellan Index
I detta sista steg görs en *Spearmans korrelations*-test mellan de olika indexen i tidsserien. Detta syftar till att visa exakt hur de olika indexen korrelerar med varandra. Datan samlas till en matris som plottas i form av en värmekarta (`heatmap`) via seaborn. Korrelationsanalysen görs via funktioner i `pandas`

In [ ]:
### Testar korrelation med Spearmans
# Välj polygon för klippning (sätt None för hela studieområdet)
clip_shp = OPEN_WETLAND_AREA[f"{area}_wetland"]  # Exempel: OPEN_WETLAND_AREA["GRM_wetland"]
colors = {"NDVI": "forestgreen", "MNDWI": "steelblue", "NDMI": "darkorange", "NDWI": "purple"}
clip_name = os.path.basename(clip_shp).replace(".shp", "") if clip_shp is not None else "hela studieområdet"

selected_for_corr = ["NDVI", "NDWI", "MNDWI", "NDMI", "EVI"]

# Läs in polygon en gång och återanvänd
clip_gdf = None
if clip_shp is not None:
    if not os.path.exists(clip_shp):
        raise FileNotFoundError(f"Kunde inte hitta polygonfil: {clip_shp}")
    clip_gdf = gpd.read_file(clip_shp)
    if clip_gdf.empty:
        raise ValueError(f"Polygonfilen är tom: {clip_shp}")
    if clip_gdf.crs is None:
        raise ValueError(f"Polygonfilen saknar CRS: {clip_shp}")

# Bygg DataFrame med medelvärden per tidssteg per index (inom polygon)
series_dict = {}
for index_name in selected_for_corr:
    da = stacks[index_name]

    if clip_gdf is not None:
        if da.rio.crs is None:
            raise ValueError(f"Stacken för {index_name} saknar CRS och kan inte klippas.")

        clip_gdf_match = clip_gdf.to_crs(da.rio.crs)
        da_use = da.rio.clip(clip_gdf_match.geometry, clip_gdf_match.crs, drop=True)
    else:
        da_use = da

    # Medelvärde per tidssteg för klippt område
    series_dict[index_name] = da_use.mean(dim=["x", "y"], skipna=True).values

df_indices = pd.DataFrame(series_dict).dropna(how="any")

print(f"Korrelationsområde: {clip_name}")
print(df_indices.head())

# Spearmans korrelationsmatris
corr_matrix = df_indices.corr(method="spearman") # .corr() metoden är hämtad från pandas och kan användas direkt på DataFrame. Alternativt kan scipy.stats.spearmanr användas för mer detaljerade resultat
print(corr_matrix)

# Visualisera som heatmap
plt.style.use("default")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ["Helvetica", "Arial", "sans-serif"]


fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    square=True,
    linewidths=0.5,
    # cbar_kws={"label": "Spearman rho"},
)
ax.set_title(f"{name}")
#ax.set_title(f"{area}")
plt.tight_layout()

# Skapa målmapp om den inte finns
output_dir_corr = r"D:\Masteruppsats\Sentinel\correlation_matrix"
os.makedirs(output_dir_corr, exist_ok=True)

# Spara som bild
out_path_fig = os.path.join(output_dir_corr, f"{name}_{clip_name}_{sensor}_correlation_matrix.png")
fig.savefig(out_path_fig, dpi=600, bbox_inches="tight")
print(f"Sparad: {out_path_fig}")

plt.show()

# Spara CSV
out_path = os.path.join(output_dir_corr, f"{name}_{clip_name}_{sensor}_correlation_matrix.csv")
corr_matrix.to_csv(out_path, encoding="utf-8-sig")
print(f"Sparad: {out_path}")